In [9]:
from rdkit import Chem
import numpy as np
import csv
import joblib
import sys
from anytree import Node, RenderTree
sys.path.append('molecule-web-app/backend/rules')  # replace with your local path
sys.path.append('molecule-web-app/backend')
# Now you can import your module as usual
from rules import execute_step
import helper
from ml_helper import make_fingerprint

In [75]:
sm_smiles = 'CO.BrBr.CC(C)=C'
end_smiles = 'Br.COC(C)(C)CBr'
sm = Chem.MolFromSmiles(sm_smiles)
end = Chem.MolFromSmiles(end_smiles)
depth_limit = 10
sm_node = Node("SM", mol=sm)

In [76]:
# Load the model
clf = joblib.load('models/v11/xg_model.pkl')

In [77]:
step_no = 0
nodes = [sm_node]
signal_break = False
unique_id = 0
while step_no < depth_limit:
    print(f'step no {step_no}')
    for int in nodes:
        nodes_to_next = []
        print(f'Layer {step_no+1}')
        try:
            product_mols, mechs, mol_inp  = execute_step("Placeholder", True, True, int.mol, False)
        except Exception as e:
            continue
        for i, product in enumerate(product_mols):
            fp = np.array(make_fingerprint(mol_inp, mechs[i]))
            # Predict using the model
            pred = clf.predict(fp.reshape(1,-1))[0]
            try:
                inchi1 = Chem.MolToInchi(product, options='Fixed-H')
                inchi2 = Chem.MolToInchi(end, options='Fixed-H')
            except Exception as e:
                print(f'Error processing product {i}: {e}')
                continue
            if inchi1 == inchi2:
                print('Found the target molecule!')
                print(f'The final step (mol {unique_id}) was predicted: {pred}')
                new_node = Node(f"Mol {unique_id}", parent=int, mol=product, mech=mechs[i])
                unique_id += 1
                signal_break = True
                break
            elif pred == 1:
                new_node = Node(f"Mol {unique_id}", parent=int, mol=product, mech=mechs[i])
                unique_id += 1
                nodes_to_next.append(new_node)
        if signal_break:
            break
    if signal_break:
        print("we breaking")
        break
    nodes = nodes_to_next
    step_no += 1

print("search is over")

       


step no 0
Layer 1
step no 1
Layer 2
Layer 2
Found the target molecule!
The final step (mol 8) was predicted: 1
we breaking
search is over


[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02] WARNING: Proton(s) added/removed

[16:20:02]

In [78]:
len(make_fingerprint(mol_inp, mechs[1]))

160

In [79]:
for pre, fill, node in RenderTree(sm_node):
    print(f"{pre}{node.name} - {Chem.MolToSmiles(node.mol)}")
    #if hasattr(node, 'mech'):
    #    print(f"{pre}Mechanism: {node.mech}")
    #else:
    #    print(f"{pre}Mechanism: None")

SM - BrBr.C=C(C)C.CO
├── Mol 0 - C=C(C)C.C[OH+]Br.[Br-]
│   ├── Mol 5 - Br.C=C(C)C.COBr
│   └── Mol 6 - C=[OH+].C[C+](C)C.[Br-].[Br-]
├── Mol 1 - BrBr.[CH2-]C(C)(C)[OH+]C
│   ├── Mol 7 - BrBr.COC(C)(C)C
│   └── Mol 8 - C[OH+]C(C)(C)CBr.[Br-]
├── Mol 2 - BrBr.C[OH+]C[C-](C)C
├── Mol 3 - CO.[Br-].[CH2+]C(C)(C)Br
└── Mol 4 - CO.C[C+](C)CBr.[Br-]


In [29]:
sm_node.children

(Node('/SM/Mol 0', mech=[8, 2, 0, 1, 2], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c4ebf40>),
 Node('/SM/Mol 4', mech=[8, 3, 0, 1, 3], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c368040>),
 Node('/SM/Mol 6', mech=[8, 4, 0, 1, 2], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c3680b0>),
 Node('/SM/Mol 8', mech=[1, 2, 1, None, 3], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c3682e0>),
 Node('/SM/Mol 10', mech=[1, 4, 1, None, 2], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c368200>))